# projects/timezone: The server

**This is my own MCP server. It offers one of each MCP primitive: A tool that converts a time between time zones, a resource that lists our offices, and a prompt that plans a meeting. It runs over Streamable HTTP, on its own or in Docker.**

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 client (client.py, probe.py)        server.py at http://127.0.0.1:8060/mcp
 ----------------------------        --------------------------------------
 tools/call      convert_time  ----&gt; convert_time(time, from_zone, to_zone)
 resources/read  zones://offices --&gt; get_offices()
 prompts/get     plan_meeting  ----&gt; plan_meeting(date, time)
</pre>

Read `../../crash-course/2-server-stdio-http-sse/client-streamable-http.ipynb` and `../../crash-course/5-run-with-docker/docker.ipynb` first. This notebook covers only what's new. `Client_Implementation_Details.ipynb` explains `client.py`.

## 1. The basics: A time zone is a set of rules, not a fixed offset

- **Offset:** How far a clock is from UTC, the world's reference clock. Pune is always UTC+05:30.
- **Time zone:** A place's rules for its offset over the year, including daylight saving time (DST), when clocks move an hour forward for the summer. New York is UTC-04:00 in summer (EDT) and UTC-05:00 in winter (EST).
- **The IANA time zone database:** The standard record of every zone's rules, with names like `Asia/Kolkata`, `Europe/London` and `America/New_York`. Python's built-in `zoneinfo` module reads it.

So converting a time needs the date, not just the hour. US clocks go back an hour on 1 November 2026:

In [1]:
from datetime import datetime
from zoneinfo import ZoneInfo

for date in ["2026-10-28", "2026-11-05"]:
    pune_time = datetime.strptime(date + " 15:00", "%Y-%m-%d %H:%M").replace(tzinfo=ZoneInfo("Asia/Kolkata"))
    new_york_time = pune_time.astimezone(ZoneInfo("America/New_York"))
    print(pune_time)
    print(new_york_time)

2026-10-28 15:00:00+05:30
2026-10-28 05:30:00-04:00
2026-11-05 15:00:00+05:30
2026-11-05 04:30:00-05:00


The same 15:00 in Pune is 05:30 in New York one week and 04:30 the next. A fixed "minus 9 hours 30 minutes" rule would be wrong for half the year. The steps:

- **`datetime.strptime(text, "%Y-%m-%d %H:%M")`:** Reads the text into a `datetime`. The pattern says where the year, month, day, hour and minute are.
- **`.replace(tzinfo=ZoneInfo("Asia/Kolkata"))`:** Attaches a zone, meaning "this 15:00 is Pune time". It doesn't change the clock time.
- **`.astimezone(ZoneInfo("America/New_York"))`:** The same moment, shown on New York's clock. This is where the DST rules are applied.
- **The printed offset:** `+05:30` for Pune, then `-04:00` before the change and `-05:00` after it.

## 2. The tool: `convert_time`

**A tool is an action the model can ask for.** This one is section 1, with the time and both zones as arguments:

<div style="font-size: 0.85em">

```python
@mcp.tool()
def convert_time(time: str, from_zone: str, to_zone: str) -> str:
    """Convert a time from one time zone to another. time is "YYYY-MM-DD HH:MM", and the zones are IANA names like "Asia/Kolkata"."""
    start = datetime.strptime(time, "%Y-%m-%d %H:%M").replace(tzinfo=ZoneInfo(from_zone))
    converted = start.astimezone(ZoneInfo(to_zone))
    return converted.strftime("%Y-%m-%d %H:%M %Z")
```

</div>

- **`strftime("%Y-%m-%d %H:%M %Z")`:** Turns the result back into text. `%Z` is the zone's short name on that date, such as `EDT` or `EST`, so the model can see which rule applied.
- **The docstring:** Becomes the tool's description. It tells the model the time format, and that the zones are IANA names, because the model fills in these arguments itself.
- **No `try/except`:** A bad zone name or time format raises an error, and FastMCP sends it back as a result with `isError: true`. Section 6 shows one.

## 3. The resource: `zones://offices`

**A resource is read-only data that the server offers under a URI. The app decides when to read it and what to do with it; the model never asks for it.** Here it's the list of our offices:

<div style="font-size: 0.85em">

```python
offices = {
    "Pune": "Asia/Kolkata",
    "London": "Europe/London",
    "New York": "America/New_York",
}


@mcp.resource("zones://offices")
def get_offices() -> str:
    """Our offices and their IANA time zones."""
    lines = []
    for city, zone in offices.items():
        lines.append(city + ": " + zone)
    return "\n".join(lines)
```

</div>

- **`"zones://offices"`:** The URI clients use to read it. `zones://` is a scheme I made up: Any `name://path` works.
- **The return value:** One line per office, joined with line breaks (`"\n"`).

## 4. The prompt: `plan_meeting`

**A prompt is a ready-made message that the server offers. The user picks it, for example from a menu in Claude Desktop, its arguments fill it in, and the host sends it to the model as the user's message.**

<div style="font-size: 0.85em">

```python
@mcp.prompt()
def plan_meeting(date: str, time: str) -> str:
    """Ask what a meeting at a given Pune time is in every office, and whether it falls in working hours."""
    return f"We want a meeting on {date} at {time} Pune time. Use the convert_time tool to find its local time in every office, and say whether it falls inside working hours, 09:00 to 18:00 local time."
```

</div>

- **`date` and `time`:** They become the prompt's arguments, both required.
- **The text:** It names the tool, so the model knows to use it.

Each primitive has a different owner:

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 primitive   who decides to use it        in this server
 ---------   ---------------------        --------------
 tool        the model                    convert_time
 resource    the app (the host's code)    zones://offices
 prompt      the user                     plan_meeting
</pre>

## 5. The host, the port, and starting it

<div style="font-size: 0.85em">

```python
mcp = FastMCP("Timezone", host=os.getenv("HOST", "127.0.0.1"), port=8060)

...

if __name__ == "__main__":
    mcp.run(transport="streamable-http")
```

</div>

- **`os.getenv("HOST", "127.0.0.1")`:** Reads the environment variable `HOST`, and uses `127.0.0.1` when it isn't set. On your computer it isn't set, so only your own machine can reach the server. The Dockerfile sets `HOST=0.0.0.0`, which a server inside a container needs (section 9). So the same `server.py` works in both places.
- **`port=8060`:** Not 8050, so it can run next to the crash course's calculator.
- **`transport="streamable-http"`:** HTTP only. Every message goes to `http://127.0.0.1:8060/mcp`.

Start it in a terminal in this folder, and leave it running while you run the cells below:

<div style="font-size: 0.85em">

```bash
conda activate MCP_env
python server.py
```

</div>

## 6. Calling all three over MCP

The same client code as in `client-streamable-http.ipynb`, now using every primitive. The last call passes a zone that doesn't exist:

In [2]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

url = "http://127.0.0.1:8060/mcp"

In [3]:
async with streamable_http_client(url) as (read, write, get_session_id):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools_result = await session.list_tools()
        offices = await session.read_resource("zones://offices")
        prompt = await session.get_prompt("plan_meeting", {"date": "2026-10-28", "time": "15:00"})
        result = await session.call_tool("convert_time", {"time": "2026-11-05 15:00", "from_zone": "Asia/Kolkata", "to_zone": "America/New_York"})
        bad_result = await session.call_tool("convert_time", {"time": "2026-11-05 15:00", "from_zone": "Asia/Kolkata", "to_zone": "Mars/Olympus_Mons"})

print(result.content[0].text)

2026-11-05 04:30 EST


The tool, as the client sees it in `tools/list`:

In [4]:
print(tools_result.tools[0].model_dump_json(indent=2, exclude_none=True))

{
  "name": "convert_time",
  "description": "Convert a time from one time zone to another. time is \"YYYY-MM-DD HH:MM\", and the zones are IANA names like \"Asia/Kolkata\".",
  "inputSchema": {
    "properties": {
      "time": {
        "title": "Time",
        "type": "string"
      },
      "from_zone": {
        "title": "From Zone",
        "type": "string"
      },
      "to_zone": {
        "title": "To Zone",
        "type": "string"
      }
    },
    "required": [
      "time",
      "from_zone",
      "to_zone"
    ],
    "title": "convert_timeArguments",
    "type": "object"
  },
  "outputSchema": {
    "properties": {
      "result": {
        "title": "Result",
        "type": "string"
      }
    },
    "required": [
      "result"
    ],
    "title": "convert_timeOutput",
    "type": "object"
  }
}


- **`inputSchema`:** Built from the type hints. Three strings, all required, because none has a default.
- **`outputSchema`:** FastMCP also describes the return value: One string, named `result`.

The resource comes back as `contents`, and the prompt as `messages`, with one message whose role is `user`:

In [5]:
print(offices.contents[0].text)
print()
print(prompt.messages[0].role)
print(prompt.messages[0].content.text)

Pune: Asia/Kolkata
London: Europe/London
New York: America/New_York

user
We want a meeting on 2026-10-28 at 15:00 Pune time. Use the convert_time tool to find its local time in every office, and say whether it falls inside working hours, 09:00 to 18:00 local time.


And the bad zone name. The server didn't crash: FastMCP caught the error and sent it back as a normal result, with `isError` set:

In [6]:
print(bad_result.isError)
print(bad_result.content[0].text)

True
Error executing tool convert_time: 'No time zone found with key Mars/Olympus_Mons'


The model reads that text like any other result, so it can tell the user what went wrong, or retry with a real zone.

## 7. The handshake on the wire, by hand

**Under the SDK, every MCP message is a small JSON-RPC message sent in an HTTP POST. `probe.py` sends the first three by hand with plain `httpx`, then checks the answers. It's a health check that needs no MCP library: It passes if the handshake works and `convert_time` is in the tool list.**

The first message is `initialize`. The client says which protocol version it speaks, and who it is:

In [7]:
import json

import httpx

headers = {"Accept": "application/json, text/event-stream"}
initialize = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2025-11-25",
        "capabilities": {},
        "clientInfo": {"name": "probe", "version": "1.0"},
    },
}
response = httpx.post(url, json=initialize, headers=headers)

print(response.status_code)
print(response.headers["Content-Type"])
print(response.text)

200
text/event-stream
event: message
data: {"jsonrpc":"2.0","id":1,"result":{"protocolVersion":"2025-11-25","capabilities":{"experimental":{},"prompts":{"listChanged":false},"resources":{"subscribe":false,"listChanged":false},"tools":{"listChanged":false}},"serverInfo":{"name":"Timezone","version":"1.27.0"}}}




- **The `Accept` header:** The client must say it can take both plain JSON and a stream. With only `application/json`, this server answers `406 Not Acceptable`.
- **The reply is a stream:** It came as `text/event-stream` (SSE) holding one event, and the JSON-RPC message is on its `data:` line. `read_message` finds that line and turns it into a dict:

In [8]:
def read_message(response):
    for line in response.text.splitlines():
        if line.startswith("data: "):
            return json.loads(line[len("data: "):])


reply = read_message(response)
print(json.dumps(reply, indent=2))

{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "protocolVersion": "2025-11-25",
    "capabilities": {
      "experimental": {},
      "prompts": {
        "listChanged": false
      },
      "resources": {
        "subscribe": false,
        "listChanged": false
      },
      "tools": {
        "listChanged": false
      }
    },
    "serverInfo": {
      "name": "Timezone",
      "version": "1.27.0"
    }
  }
}


- **`"id": 1`:** The same id as the request, so the client knows which request this answers.
- **`protocolVersion`:** The server agrees to 2025-11-25.
- **`capabilities`:** What the server offers: Tools, resources and prompts. `listChanged: false` means it won't announce changes to those lists.
- **`serverInfo`:** The name from `FastMCP("Timezone")`, and the SDK's version.

The server also sent a session ID, in a header. Every later request carries it back, along with the protocol version. Then the client confirms the handshake with `notifications/initialized`:

In [9]:
headers["Mcp-Session-Id"] = response.headers["Mcp-Session-Id"]
headers["MCP-Protocol-Version"] = "2025-11-25"

response = httpx.post(url, json={"jsonrpc": "2.0", "method": "notifications/initialized"}, headers=headers)
print(response.status_code)

202


`202 Accepted`, with an empty body: A notification has no `id`, so it never gets a reply. From here on, the client can make requests, like `tools/list`:

In [10]:
response = httpx.post(url, json={"jsonrpc": "2.0", "id": 2, "method": "tools/list"}, headers=headers)
tools = read_message(response)["result"]["tools"]
print(response.status_code)
print(tools[0]["name"])

200
convert_time


Finally, `DELETE` ends the session:

In [11]:
response = httpx.delete(url, headers=headers)
print(response.status_code)

200


`probe.py` is exactly these steps in one script, with PASS or FAIL at the end:

<div style="font-size: 0.85em">

```python
import json

import httpx

url = "http://127.0.0.1:8060/mcp"
headers = {"Accept": "application/json, text/event-stream"}


def read_message(response):
    for line in response.text.splitlines():
        if line.startswith("data: "):
            return json.loads(line[len("data: "):])


initialize = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2025-11-25",
        "capabilities": {},
        "clientInfo": {"name": "probe", "version": "1.0"},
    },
}
response = httpx.post(url, json=initialize, headers=headers)
server_info = read_message(response)["result"]["serverInfo"]
print("initialize:", response.status_code, server_info["name"])

headers["Mcp-Session-Id"] = response.headers["Mcp-Session-Id"]
headers["MCP-Protocol-Version"] = "2025-11-25"

response = httpx.post(url, json={"jsonrpc": "2.0", "method": "notifications/initialized"}, headers=headers)
print("notifications/initialized:", response.status_code)

response = httpx.post(url, json={"jsonrpc": "2.0", "id": 2, "method": "tools/list"}, headers=headers)
tool_names = []
for tool in read_message(response)["result"]["tools"]:
    tool_names.append(tool["name"])
print("tools/list:", response.status_code, tool_names)

httpx.delete(url, headers=headers)

if response.status_code == 200 and "convert_time" in tool_names:
    print("PASS")
else:
    print("FAIL")
```

</div>

With the server running, `python probe.py` printed:

<div style="font-size: 0.85em">

```text
initialize: 200 Timezone
notifications/initialized: 202
tools/list: 200 ['convert_time']
PASS
```

</div>

This is how an automated check can test any MCP server from outside: Send the handshake, then assert HTTP 200 and the expected tool.

## 8. What the server saw

While the cells above ran, the server's terminal printed:

<div style="font-size: 0.85em">

```text
Created new transport with session ID: 7b1f126bc101490b94fb9492ef8d1400
INFO:     127.0.0.1:64476 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64477 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:64478 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64479 - "POST /mcp HTTP/1.1" 200 OK
Processing request of type ListToolsRequest
INFO:     127.0.0.1:64480 - "POST /mcp HTTP/1.1" 200 OK
Processing request of type ReadResourceRequest
INFO:     127.0.0.1:64481 - "POST /mcp HTTP/1.1" 200 OK
Processing request of type GetPromptRequest
INFO:     127.0.0.1:64482 - "POST /mcp HTTP/1.1" 200 OK
Processing request of type CallToolRequest
INFO:     127.0.0.1:64483 - "POST /mcp HTTP/1.1" 200 OK
Processing request of type CallToolRequest
Terminating session: 7b1f126bc101490b94fb9492ef8d1400
INFO:     127.0.0.1:64484 - "DELETE /mcp HTTP/1.1" 200 OK
Created new transport with session ID: b4ec2468197043e5a09bdc9dda905da0
INFO:     127.0.0.1:64485 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64486 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:64487 - "POST /mcp HTTP/1.1" 200 OK
Processing request of type ListToolsRequest
Terminating session: b4ec2468197043e5a09bdc9dda905da0
INFO:     127.0.0.1:64488 - "DELETE /mcp HTTP/1.1" 200 OK
```

</div>

There are two sessions: The SDK's in section 6 (with its extra `GET` stream), and the hand-made one in section 7 (no `GET`, because we never opened one). Each `Processing request` line names the MCP request behind the POST above it.

## 9. Running it in Docker

The Dockerfile is the one from Part 5, with one new line, `ENV HOST=0.0.0.0`:

<div style="font-size: 0.85em">

```dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install -r requirements.txt

COPY server.py .

ENV HOST=0.0.0.0
EXPOSE 8060

CMD ["python", "server.py"]
```

</div>

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
 FROM python:3.12-slim     a small official image with Python 3.12
 WORKDIR /app              work in /app inside the image
 COPY requirements.txt .   copy the package list first ...
 RUN pip install ...       ... and install it (a cached layer)
 COPY server.py .          then the server (the clients stay outside)
 ENV HOST=0.0.0.0          sets HOST, so os.getenv("HOST") returns 0.0.0.0
 EXPOSE 8060               notes the port (doesn't publish it)
 CMD ["python", ...]       what the container runs when it starts
</pre>

`requirements.txt` pins what MCP_env has:

<div style="font-size: 0.85em">

```text
mcp==1.27.0
pydantic-settings==2.14.0
```

</div>

There's no `tzdata` package in it. That was checked: A plain `python:3.12-slim` container already has the time zone database, because `ZoneInfo("Asia/Kolkata")` works there with nothing installed. Python's `zoneinfo` reads that system copy first.

With Docker Desktop running, build the image and start a container. Stop the local server first, because both use port 8060:

<div style="font-size: 0.85em">

```bash
docker build -t timezone-mcp .
docker run --rm --name timezone-mcp -p 127.0.0.1:8060:8060 timezone-mcp
```

</div>

The key lines of the build, and `docker ps` in a second terminal:

<div style="font-size: 0.85em">

```text
#5 [1/5] FROM docker.io/library/python:3.12-slim@sha256:a0f57064cbdcd6285fa45f51237ab0b35ea8ebaed06a1f5973e89e19dfb86704
#6 [2/5] WORKDIR /app
#7 [3/5] COPY requirements.txt .
#8 [4/5] RUN pip install -r requirements.txt
#9 [5/5] COPY server.py .
#10 naming to docker.io/library/timezone-mcp:latest done

IMAGE          STATUS                  PORTS                      NAMES
timezone-mcp   Up Less than a second   127.0.0.1:8060->8060/tcp   timezone-mcp
```

</div>

The URL doesn't change, so `python probe.py` works on the container as it is, and printed `PASS` with the same four lines as in section 7. The container's log (`docker logs timezone-mcp`) shows the server on `0.0.0.0`, and the requests arriving through Docker's gateway, `172.17.0.1`:

<div style="font-size: 0.85em">

```text
INFO:     Started server process [1]
INFO:     Waiting for application startup.
StreamableHTTP session manager started
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8060 (Press CTRL+C to quit)
Created new transport with session ID: dfc4728e7b8f4b8bba3342292d13d8fb
INFO:     172.17.0.1:49308 - "POST /mcp HTTP/1.1" 200 OK
INFO:     172.17.0.1:49324 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     172.17.0.1:49336 - "POST /mcp HTTP/1.1" 200 OK
Processing request of type ListToolsRequest
Terminating session: dfc4728e7b8f4b8bba3342292d13d8fb
INFO:     172.17.0.1:49346 - "DELETE /mcp HTTP/1.1" 200 OK
```

</div>

`client.py` also ran on the container, and got the same four conversions as on the local server. Stop it with `Ctrl+C` or `docker stop timezone-mcp`.

## Say it out loud

> "I built my own MCP server with FastMCP: a time zone converter with one of each primitive. The `convert_time` tool, which the model calls, uses Python's `zoneinfo`, so daylight saving is handled: 15:00 in Pune is 05:30 in New York in late October and 04:30 a week later. The `zones://offices` resource is data the app reads, and the `plan_meeting` prompt is a template the user picks. It runs over Streamable HTTP on port 8060. The same file runs in Docker, because the host comes from an environment variable: 127.0.0.1 locally and 0.0.0.0 in the container. I check it with a probe script that sends the raw JSON-RPC handshake with plain httpx, and asserts HTTP 200 and the `convert_time` tool, against both the local server and the container. I built it after my internship, when I moved my MCP work from stdio and SSE to Streamable HTTP."